# 在一張 T4 上微調 110 億參數的視覺語言模型

**天文影像描述 · 上機實作本**

---

### 開始前，三件事

1. **執行階段 → 變更執行階段類型 → T4 GPU**
2. **一格一格按，不要用「全部執行」。**
3. 標 🔧 的是**你要動手改的格子**；標 🐞 的表示**這裡刻意留了 bug**——先照原樣跑，別急著修。

> 這本刻意保留了原始專案的四個 bug。**它們是教材，不是疏漏**——你會在 M5-2 到 M6-2 親手把它們抓出來。

In [ ]:
!nvidia-smi

## 課前｜取得資料集

原始專案從作者私人的 Google Drive 讀 `astronomy_dataset.zip`，那份沒有公開。
這裡改從 Hugging Face 抓同一份資料，還原成 notebook 需要的磁碟結構：

```
astronomy_dataset/
    data.json      [{"image_id","text","image":"images/001.jpg"}, ...]
    images/001.jpg ...
```

In [ ]:
!pip install -q datasets

import json, os, re, random, zipfile
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from datasets import Dataset, load_dataset

BASE = "/content/astronomy_dataset"

def build_local_dataset(repo="AIOmarRehan/space-multimodal-dataset"):
    """把 HF 上的資料還原成 data.json + images/ 的結構。"""
    if os.path.exists(os.path.join(BASE, "data.json")):
        print("已存在，跳過下載")
        return
    ds = load_dataset(repo, split="train")
    print("HF 欄位：", ds.column_names, "| 筆數：", len(ds))
    os.makedirs(os.path.join(BASE, "images"), exist_ok=True)
    records = []
    for row in ds:
        iid = str(row["image_id"]).strip()
        if iid.isdigit():
            iid = iid.zfill(3)
        rel = f"images/{iid}.jpg"
        row["image"].convert("RGB").save(os.path.join(BASE, rel), quality=95)
        records.append({"image_id": iid, "text": row["text"], "image": rel})
    with open(os.path.join(BASE, "data.json"), "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print("完成：", len(records), "筆")

build_local_dataset()

with open(os.path.join(BASE, "data.json")) as f:
    df = pd.DataFrame(json.load(f))

print(df.shape)
df.head()

---
## M1｜載入模型與掛上 LoRA

先按下面兩格，**下載約 5 GB，要跑 3–5 分鐘**。等的時候先讀這段。

### 為什麼 110 億參數塞得進 15 GB？

```
權重（fp16）    10.74e9 x 2 bytes            = 21.5 GB   <- T4 只有 14.7 GB，載不進去
權重（4-bit）   10.74e9 x 0.5 bytes          =  5.4 GB   <- 進得去了
優化器狀態      AdamW 每個參數要多存 2 個數
                全量微調：10.74e9 x 8 bytes  = 85.9 GB   <- 想都別想
                LoRA + 8-bit：67.2e6 x 2     =  0.13 GB
```

**關鍵在第三行。** 很多人以為訓練的記憶體瓶頸是權重，其實是**優化器**——它比權重本身還大四倍。

**LoRA 省的不是權重的記憶體，是優化器的記憶體。** 因為只有 6700 萬個參數需要被更新，這一項直接從 85.9 GB 掉到 0.13 GB。

### LoRA 在做什麼

```
原本：  h = W . x            W 是大矩陣，凍結不動
LoRA：  h = W . x  +  B.A.x
                   |______|  A、B 兩個很瘦的矩陣，加起來不到原本的 1%
```

假設是：把一個已經很強的模型微調到新任務，**權重的「變化量」其實資訊量很少**，少到可以用兩個瘦矩陣相乘表示。所以不動 W，只訓練 A 和 B。

`r`（秩）就是那兩個矩陣有多瘦——越大能學越多，但在 250 筆資料上也越容易過擬合。

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import torch

# 護欄：不花時間，但擋掉這格最常見的兩種當機（要放在 import unsloth 前面，見下一格）
assert torch.cuda.is_available(), "沒有 GPU：執行階段 → 變更執行階段類型 → T4 GPU，然後從第一格重跑"
gpu_used_gb = torch.cuda.memory_allocated() / 1024**3
assert gpu_used_gb < 1, f"GPU 已被佔用 {gpu_used_gb:.1f} GB —— 這格不能重跑，請先「重新啟動工作階段」"

from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit",   # 明寫 4-bit 版本，理由見下一格
    load_in_4bit = True,                      # 沒有這行，T4 直接 OOM
    use_gradient_checkpointing = "unsloth",   # 用計算時間換記憶體
)

跑完注意啟動橫幅的兩行：`Tesla T4 ... Max memory: 14.741 GB` 和 `Bfloat16 = FALSE`
（T4 是 Pascal 之後的 Turing 架構，**不支援 bfloat16**，所以我們在 fp16 上訓練）。

上面那兩行 `assert` 是**護欄**——第一行擋「忘了切 GPU」，第二行擋「這格重跑第二次」。
它們**排在 `from unsloth import ...` 前面**是刻意的：unsloth 自己 import 的時候就會檢查 GPU、
丟出它自己的英文錯誤，而那個 import 要跑快一分鐘。檢查寫在後面的話，在「忘了切 T4」
這個它唯一該作用的情境裡**永遠輪不到它講話**。
**護欄要放在最早能講話的位置**——晚一步的護欄等於沒有護欄。
GPU 上已經有一個模型時，第二個就載不進去，而且 `torch.cuda.empty_cache()` **救不回來**
（那是活著的張量，不是快取），只能重新啟動工作階段。護欄這個習慣 M5-3 會再出現一次。

模型名字**明寫 `-bnb-4bit`**：原始 notebook 寫的是 `unsloth/Llama-3.2-11B-Vision-Instruct`，
靠 unsloth 內部一張對照表自動幫你換成 4-bit 版。能動，但多了一層看不見的依賴。
寫死名字，你載到的是什麼一目了然，也才對得上前面說的「下載約 5 GB」。

> 這其實就是 M5-4 那條守則提早出現：**明確永遠勝過依賴預設值。**

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,   # 視覺編碼器要不要動
    finetune_language_layers   = True,   # 語言模型要不要動
    finetune_attention_modules = True,   # attention 的 Q/K/V/O 要不要掛 LoRA
    finetune_mlp_modules       = True,   # FFN 要不要掛 LoRA
    r = 16,           # 薄膜有多厚
    lora_alpha = 16,  # 經驗法則：alpha >= r
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

**記下這行**：`Trainable parameters = 67,174,400 of 10,737,395,235 (0.63% trained)`

6700 萬 / 107 億 = **0.63%**。這就是今天這件事能在免費顯卡上做到的原因。

---
## M2-1｜多模態對話格式

全課最需要背的格式。注意三件事：`content` 是 **list** 不是字串；圖片放**路徑字串**（誰讀成圖片？下面的 collator）；**assistant 那段就是我們要教模型說的話**。

In [ ]:
INSTRUCTION = "You are an expert astronomer. Describe accurately what you see in this image."

def convert_to_conversation(sample):
    return {"messages": [
        {"role": "user", "content": [
            {"type": "text",  "text": INSTRUCTION},
            {"type": "image", "image": os.path.join(BASE, sample["image"])},
        ]},
        {"role": "assistant", "content": [
            {"type": "text", "text": sample["text"]},
        ]},
    ]}

import pprint
pprint.pprint(convert_to_conversation(df.iloc[0]))

---
## M2-2｜Lab 2-A：chat template 與 image token

看模型**真正吃到的字串**長什麼樣。一定要用 `repr`，不然特殊標記看不出來。

In [ ]:
messages = [{"role": "user", "content": [
    {"type": "image"},
    {"type": "text", "text": INSTRUCTION},
]}]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
print(repr(input_text))

找出 `<|image|>` 在哪裡。它是一個**佔位符**——模型跑的時候，會在那個位置把影像編碼器算出來的特徵接進去（cross-attention）。

`add_generation_prompt=True` 會在結尾補上「輪到 assistant 講話了」。**訓練時不要加，推論時一定要加。**

---
## M2-3｜tokenizer 呼叫 ＋ 微調前的對照組

**這格是全課最容易寫錯的一行**，也是等一下驗收成果用的「之前」。

```python
tokenizer(image, input_text, ...)
         # ↑圖片在前，文字在後，而且是位置引數。寫反了不會噴錯，只會給你垃圾。
```

In [ ]:
from transformers import TextStreamer

FastVisionModel.for_inference(model)

sample_row  = df.iloc[0]
image       = Image.open(os.path.join(BASE, sample_row["image"])).convert("RGB")
input_text  = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

inputs = tokenizer(
    image,                      # 第一個位置引數：圖片
    input_text,                 # 第二個位置引數：文字
    add_special_tokens=False,   # template 已經加過開頭標記了，再加會變兩份
    return_tensors="pt",
).to("cuda")

print("【微調前】原廠模型的描述風格：\n")
_ = model.generate(**inputs, streamer=TextStreamer(tokenizer, skip_prompt=True),
                   max_new_tokens=128, use_cache=True,
                   do_sample=True, temperature=1.2, min_p=0.1)   # 明寫 do_sample，見 M5-4
print("\n【資料集要的風格】\n", sample_row["text"])

**把這兩段對照著看。** 原廠會寫出文情並茂的長散文，我們的資料集要的是一句話。

> 微調不是在教模型新知識，是在教模型**新的說話方式**。

（此刻 LoRA 剛掛上、B 矩陣還是全零，所以上面就是**純原廠**的表現。）

---
## M2-4｜Lab 2-B：collator 到底做了什麼

前面說「loss 只算 assistant 那一段」。**不要相信，自己驗證。**

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
collator = UnslothVisionDataCollator(model, tokenizer)

demo  = [convert_to_conversation(r) for _, r in df.head(2).iterrows()]
batch = collator(demo)

labels = batch["labels"][0]
print("batch 的 key：", list(batch.keys()))
print("序列長度：", tuple(labels.shape))
print("被遮掉（-100）的比例：%.1f%%" % ((labels == -100).float().mean().item() * 100))

kept = labels[labels != -100]
print("\n實際會被學的內容：")
print(tokenizer.decode(kept))

最後印出來的應該**只有那句 caption**，前面所有 prompt 都不見了——`-100` 是 PyTorch 交叉熵的慣例，代表「這個位置不算分」。

遮掉的比例大概九成以上，因為**圖片本身就佔了大量 token**。這代表 `max_length=2048` 裡留給文字的空間，比你想的少很多。

---
## M3-1｜資料集體檢

In [ ]:
print("筆數：", len(df), "| 不重複 caption：", df["text"].nunique())
print("缺失值：\n", df.isnull().sum().to_string())
print("空字串：", (df == "").sum().sum())
print("重複 caption：", df.duplicated(subset="text").sum())

df["text_length"] = df["text"].apply(len)
df["word_count"]  = df["text"].apply(lambda x: len(x.split()))
print("平均長度：%.3f 字元 / %.3f 詞" % (df["text_length"].mean(), df["word_count"].mean()))

plt.figure(figsize=(8,4))
plt.hist(df["word_count"], bins=20, color="skyblue", edgecolor="black")
plt.title("Caption Word Count Distribution"); plt.xlabel("words"); plt.show()

零缺失、零重複、全部乾淨——**這是一份乾淨到不真實的資料集**，你們畢業後拿到的不會長這樣。

但作者仍然把檢查全寫了一遍。這不是浪費，是專業：**檢查是免費的，清洗不是。**

---
## M3-2｜標籤是「猜」出來的

In [ ]:
def detect_label(text):
    t = text.lower()
    if   "earth"  in t:                          return "Earth"
    elif "mars"   in t and "rover" not in t:     return "Mars"
    elif "hubble" in t:                          return "Hubble"
    elif "milky"  in t:                          return "Milky Way"
    elif "rover"  in t or "laboratory" in t:     return "Mars Rover"
    else:                                        return "Unknown"

df["label"] = df["text"].apply(detect_label)
print(df["label"].value_counts().to_string())

print("\n被判成 Unknown 的前 5 筆：")
for t in df[df["label"] == "Unknown"]["text"].head(5):
    print(" -", t[:70])

兩件事：

1. 這是 **if-elif 鏈，順序決定一切**。一句話同時提到 earth 和 hubble，會被判成 Earth。
2. 去看 Unknown 那幾筆——**它們其實全都是地球**，只是講「亞洲」「北美」「非洲」，沒出現 earth 這個字。

順帶一提，專案的 `README.md` 寫 `Mars: 54`、而且**沒有 Unknown 這一類**，跟你剛跑出來的不一樣。

> **守則一：相信執行輸出，不要相信 README。**

---
## M3-3｜🔧 Lab 3：清洗的代價

**這格會覆蓋掉 `df['text']`，也就是我們的訓練目標。先看清楚它做了什麼。**

In [ ]:
def clean_caption(text):
    text = text.lower()                        # 全部轉小寫
    text = re.sub(r"[^a-z\s]", "", text)       # 只留英文字母和空白 → 數字、標點全殺
    text = re.sub(r"\s+", " ", text).strip()
    return text

before = df["text"].iloc[1]
print("原始    :", before)
print("激進清洗:", clean_caption(before))
print("保守清洗:", re.sub(r"\s+", " ", before).strip())

三件事被改掉：全變小寫、句號消失、`wide-angle` → `wideangle`（連字號被刪掉而不是換成空白，造出一個不存在的英文字）。

**這個清洗有錯嗎？** 如果目的是詞頻統計，它完全正確。但這裡是**訓練目標**——等於在跟模型說「你講話不准用大寫、不准有標點、不准講數字」。

> **守則二：分析用的清洗 ≠ 訓練用的清洗。**

等一下你會看到模型**真的學會了**——連我們的 bug 一起學走。

In [ ]:
# 照原專案的做法覆蓋訓練目標（等一下看評估結果就知道代價）
df["text"] = df["text"].apply(clean_caption)
df = df.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
print("清洗後剩：", len(df), "筆")
df["text"].head(3).tolist()

---
## M3-4｜影像檢查

In [ ]:
def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()          # 只檢查檔頭結構，不會真的解碼整張圖
        return True
    except Exception as e:
        print("壞檔：", path, e)
        return False

df["valid_image"] = [is_valid_image(os.path.join(BASE, p)) for p in df["image"]]
print("無效影像：", (~df["valid_image"]).sum())
df = df[df["valid_image"]].reset_index(drop=True)
print("剩餘：", len(df))

兩個 Pillow 細節：`verify()` 執行完**那個 file object 就報廢了**（要用得重新 open）；而且它**只檢查檔頭**，檔頭正常但資料截斷的圖會過關，訓練到那筆才炸——要真的確認得用 `img.load()`。

---
## M4｜建立 trainer 並開始訓練

**🐞 這幾格藏著一個 bug。先照原樣跑，不要修——M5-3 會回來抓它。**

最下面三個參數是所有人第一次做 vision SFT 都會卡住的地方，它們的共同主題是：
**把 TRL 為純文字設計的自動化關掉，改成手動接管。**

In [ ]:
# M5-3 會回來檢查這一行。先照跑。
hf_dataset = Dataset.from_list([convert_to_conversation(r) for _, r in df.iterrows()])
print(hf_dataset)

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = hf_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,   # 實際同時進 GPU 的樣本數
        gradient_accumulation_steps = 4,   # 累積 4 次才更新 → 有效 batch = 8
        warmup_steps = 5,
        max_steps = 30,                    # 30 × 8 = 240 < 250 → 連一個 epoch 都沒跑完
        learning_rate = 2e-4,              # 比全量微調大十倍，因為 LoRA 的 B 是零初始化
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        remove_unused_columns = False,               # 別刪欄位，collator 要用圖片路徑
        dataset_text_field = "",                     # 沒有文字欄位，別找了
        dataset_kwargs = {"skip_prepare_dataset": True},  # 別預先 tokenize
        max_length = 2048,
    ),
)

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved()/1024/1024/1024, 3)
max_memory       = round(gpu_stats.total_memory/1024/1024/1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

### 按下去，約 11.5 分鐘

跑的時候注意 log 的 `Num examples = ?`——**這個數字等一下會變成證據。**

In [ ]:
trainer_stats = trainer.train()

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved()/1024/1024/1024, 3)
print(f"訓練耗時：{trainer_stats.metrics['train_runtime']:.1f} 秒"
      f" = {trainer_stats.metrics['train_runtime']/60:.2f} 分鐘")
print(f"峰值記憶體：{used_memory} GB / {max_memory} GB"
      f" ({used_memory/max_memory*100:.1f}%)")
print(f"其中訓練本身多吃：{round(used_memory-start_gpu_memory,3)} GB")

十 GB 裡有八點五是「把模型放進去」，訓練本身只用了一點五。
**在這個設定下，瓶頸是模型大小，不是 batch size。**

另外注意：整本 notebook **沒有 validation loss**，所以你看不出來有沒有過擬合。這是作業 A4。

---
## M5-1｜評估：先看分數，然後懷疑它

**🐞 下面幾格藏著兩個 bug。照原樣跑，先不要修——M5-2 和 M5-3 會回來抓。**

In [ ]:
from sklearn.model_selection import train_test_split

# M5-3 會回來看這一行
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
test_dataset = Dataset.from_list([convert_to_conversation(r) for _, r in test_df.iterrows()])
print("測試集：", len(test_dataset), "筆")

In [ ]:
FastVisionModel.for_inference(model)

predictions, references = [], []
raw_outputs, prompt_lens = [], []   # 留給 M5-2 做「同一批輸出、只換 decode」的對照
for i in range(len(test_dataset)):
    s = test_dataset[i]
    image_path = s["messages"][0]["content"][1]["image"]
    gt         = s["messages"][1]["content"][0]["text"]
    image      = Image.open(image_path).convert("RGB")

    msgs = [{"role":"user","content":[{"type":"image"},{"type":"text","text":INSTRUCTION}]}]
    it   = tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
    inp  = tokenizer(image, it, add_special_tokens=False, return_tensors="pt").to("cuda")

    out = model.generate(**inp, max_new_tokens=128, temperature=1.0, use_cache=True)

    raw_outputs.append(out[0])                       # 原始張量留著，M5-2 要用
    prompt_lens.append(inp["input_ids"].shape[1])    # prompt 佔了幾個 token
    predictions.append(tokenizer.decode(out[0], skip_special_tokens=True).strip())  # ← M5-2 會回來看這行
    references.append(gt.strip())

print("完成", len(predictions), "筆")

In [ ]:
!pip install -q evaluate rouge_score
from evaluate import load

bleu, rouge = load("bleu"), load("rouge")
bleu.add_batch(predictions=predictions, references=[[r] for r in references])
rouge.add_batch(predictions=predictions, references=references)
bleu_score, rouge_score_ = bleu.compute(), rouge.compute()

print("BLEU :", bleu_score)
print("ROUGE:", rouge_score_)

In [ ]:
for i in range(3):
    print(f"--- 第 {i+1} 筆 ---")
    print("Predicted:", predictions[i])
    print("Reference:", references[i])
    print()

**矛盾出現了**：人看起來不錯（認得出哈伯、認得出非洲，而且學會了小寫無標點的風格），但 BLEU 很難看。

遇到這種矛盾只有兩種可能：**指標不適合**，或**評估程式有 bug**。今天兩個都是。

---
## M5-2｜🐞 Bug #1：Prompt 回音

看上面 BLEU 輸出的 `length_ratio`。它大概是 **2 左右**——我們產生的文字是標準答案的兩倍長。

可是剛剛那三個範例長度明明差不多。**回去看完整的 Predicted 字串**，它把題目整個抄了一遍才寫答案。

原因：`model.generate` 回傳的是**輸入＋輸出的完整序列**，直接 decode 整條當然把 prompt 也印出來了。

In [ ]:
# 驗算：扣掉 prompt 回音之後，長度比例應該回到 1 附近
prompt_words = len(("user " + INSTRUCTION + " assistant").split())
n = len(predictions)
print("目前 length_ratio      :", round(bleu_score["length_ratio"], 4))
print("translation_length     :", bleu_score["translation_length"])
print("reference_length       :", bleu_score["reference_length"])
print(f"估計污染 ≈ {n} 筆 × {prompt_words} 詞 = {n*prompt_words}")
print("扣掉之後 ≈", bleu_score["translation_length"] - n*prompt_words,
      "（對比 reference", bleu_score["reference_length"], "）")

### 🔧 Lab 5-A：修好它，重算分數

注意這次的做法：**我們不重新生成，只把同一批輸出換個方式 decode。**
這樣前後唯一的差別就是有沒有切掉 prompt，分數的變化才能歸因到這個 bug——
如果順手把解碼參數也改了，你就分不清是哪一項造成的。**這正是做對照實驗的基本功。**

用 token 長度切是精確的；另一種寫法 `split("assistant")[-1]` 比較脆弱，
模型輸出裡剛好出現 "assistant" 就會把答案切斷。

In [ ]:
def generate_caption(image, prompt=INSTRUCTION, **gen_kwargs):
    """把推論邏輯包成「一個」函式 —— 評估和部署都呼叫它，才不會兩邊不一致。
    這就是作業 A3 要你做的事；下面的 Gradio 與重新評估都應該用它。"""
    msgs = [{"role":"user","content":[{"type":"image"},{"type":"text","text":prompt}]}]
    it   = tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
    inp  = tokenizer(image, it, add_special_tokens=False, return_tensors="pt").to("cuda")
    out  = model.generate(**inp, use_cache=True, **gen_kwargs)
    prompt_len = inp["input_ids"].shape[1]                      # ← 修法一
    return tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()

# 關鍵：不重新生成，直接把「同一批 output」換個方式 decode。
# 這樣前後唯一的差別就是有沒有切掉 prompt —— 沒有其他變因混進來。
fixed = [tokenizer.decode(o[pl:], skip_special_tokens=True).strip()
         for o, pl in zip(raw_outputs, prompt_lens)]

bleu2, rouge2 = load("bleu"), load("rouge")
bleu2.add_batch(predictions=fixed, references=[[r] for r in references])
rouge2.add_batch(predictions=fixed, references=references)
b2, r2 = bleu2.compute(), rouge2.compute()

print("修正前 BLEU %.4f | length_ratio %.4f" % (bleu_score["bleu"], bleu_score["length_ratio"]))
print("修正後 BLEU %.4f | length_ratio %.4f" % (b2["bleu"], b2["length_ratio"]))
print("\nROUGE-1  %.4f → %.4f" % (rouge_score_["rouge1"], r2["rouge1"]))
print("\n修正後第一筆：", fixed[0])
print("（對照）bug 版第一筆：", predictions[0][:90], "...")

> **守則四：評估路徑跟部署路徑，必須是同一段程式碼。**
>
> 這個專案的 `app.py` **有**處理 prompt 回音，評估時卻忘了。兩份 code 遲早不一致，而不一致的那天，你會用錯誤的數字做決策。

---
## M5-3｜🐞 Bug #2：測試集根本沒被隔離

往回看 M4：我們用**整個 `df`** 建 `hf_dataset` 去訓練（訓練 log 印的是 `Num examples = 250`），
然後 M5-1 又從**整個 `df`** 切測試集。

所以評估用的每一筆，模型在訓練時**都看過**。這叫 **data leakage**。

In [ ]:
train_imgs = {m["messages"][0]["content"][1]["image"] for m in hf_dataset}
test_imgs  = {m["messages"][0]["content"][1]["image"] for m in test_dataset}
leaked = train_imgs & test_imgs

print(f"訓練集 {len(train_imgs)} 筆 / 測試集 {len(test_imgs)} 筆")
print(f"洩漏 {len(leaked)} 筆 —— 佔測試集 {len(leaked)/len(test_imgs)*100:.0f}%")

### 🔧 Lab 5-B：正確切分 ＋ 加上護欄

`assert` 這一行叫 **guardrail**。它不花時間，但會在你犯錯的**那一秒**攔下你，而不是等你把論文投出去。

**你們以後每一個 ML 專案都應該有這一行。**

In [ ]:
full   = Dataset.from_list([convert_to_conversation(r) for _, r in df.iterrows()])
split1 = full.train_test_split(test_size=0.2, seed=42)
split2 = split1["test"].train_test_split(test_size=0.5, seed=42)

train_dataset = split1["train"]
valid_dataset = split2["train"]
clean_test    = split2["test"]
print(f"Train {len(train_dataset)} / Valid {len(valid_dataset)} / Test {len(clean_test)}")

tr = {m["messages"][0]["content"][1]["image"] for m in train_dataset}
te = {m["messages"][0]["content"][1]["image"] for m in clean_test}
assert not (tr & te), f"洩漏了 {len(tr & te)} 筆！"
print("無洩漏，train/test 完全分離")

**重新訓練是作業 A1**（要再花 11 分鐘，這裡先不做）。

現在回答一個問題：我們的 BLEU，是**高估**還是**低估**了真實效能？

| 因素 | 方向 |
|---|---|
| 資料洩漏（模型見過答案） | **高估** |
| Prompt 回音（分母被污染） | **低估** |
| 只訓練 0.96 個 epoch | **低估** |

三個方向不同、量級不明——**所以這個數字根本不能用**。

> **守則五：一個有 bug 的評估數字，比沒有數字更危險。**
> 沒有數字時你知道自己不知道；有錯誤數字時，你以為自己知道。

---
## M5-4｜🐞 Bug #3（小號）：傳了 temperature，卻沒傳 do_sample

M5-1 的評估迴圈寫 `temperature=1.0`，但**沒有傳 `do_sample`**。

`temperature` / `top_p` / `min_p` **只有在 `do_sample=True` 時才有作用**。那預設值來自哪裡？來自模型自己帶的設定檔——**同一行程式碼換個模型，行為就變了**。所以：自己印出來看。

In [ ]:
print(model.generation_config)

**明確永遠勝過依賴預設值。** 對照 `app.py`，它明寫了 `do_sample=True`。

另外：這個 notebook 推論用 temperature 1.2–1.5，對「準確描述一張圖」來說**太高了**。
溫度高 = 更有創意 = 更容易亂講。captioning 應該用低溫甚至 greedy（作業 B3）。

最後，就算三個 bug 全修好，**BLEU 對 captioning 本來就不適合**——一張圖有無限多種正確描述方式，
兩句都對但用字不重疊，BLEU 照樣給零分。該用 CIDEr / CLIPScore / LLM-as-judge（作業 B4）。

---
## M6-1｜存檔（只存 adapter）

兩種存法差三個數量級：

| 方式 | 體積 | 用途 |
|---|---|---|
| **只存 adapter** | 幾十 MB | **預設選這個**，HF Space 就是這樣做的 |
| 合併成完整模型 | 20 GB+ | 要餵給 vLLM 才需要；4-bit 合併還會有量化誤差 |

原專案的匯出區有四格標了 `YOUR OWN RISK`，而且合併那格**沒有包 `if False`**——
Run All 會直接觸發。**防呆做了一半，比沒做更危險。**

合併完之後還有下一步——把它載回來。那一步的代價是 M6-2。

In [ ]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("已存 adapter：")
!du -sh lora_model

想推上 Hugging Face 的話（需要 token，選做）：

```python
from huggingface_hub import login; login()
model.push_to_hub("你的帳號/astro-llama-lora", tokenizer=tokenizer)
```

---
## M6-2｜🐞 Bug #4：合併完的模型，載不回 T4

上一格你存的是 adapter，幾十 MB。**原專案不是這樣做的。**

它在匯出區先 `save_pretrained_merged()` 把 LoRA 合併回權重、寫成一份完整模型，
然後再把合併結果**載回來部署**：

```python
model, tokenizer = FastVisionModel.from_pretrained(
    "/content/unsloth_finetune",
    load_in_4bit=False,            # 16-bit instead of 4-bit
    use_gradient_checkpointing="unsloth",
    device_map="auto"              # auto splits layers across GPU and CPU
)
```

在 T4 上按下去，它會跑一分多鐘——看起來很像在正常載入——然後噴這個：

```
OutOfMemoryError: CUDA out of memory. Tried to allocate 28.00 MiB.
GPU 0 has a total capacity of 14.56 GiB of which 9.81 MiB is free.
Of the allocated memory 14.40 GiB is allocated by PyTorch,
and 21.48 MiB is reserved by PyTorch but unallocated.
If reserved but unallocated memory is large try setting
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.
```

### 🔧 Lab 6：只看訊息，不看 code

先別急著把最後那行 `PYTORCH_CUDA_ALLOC_CONF` 複製去貼上。看著上面那段訊息回答三題：

1. 最後一行建議的**前提**是什麼？（它是個條件句，「**如果** …」）
   往回找那個量，它是多少？前提成立嗎？
2. 它掛掉是因為要不到 **28 MiB**。所以我們是「差 28 MiB 就成功了」嗎？
3. `device_map="auto"` 的註解說「auto splits layers across GPU and CPU」。
   如果它真的把層分去 CPU 了，還會 OOM 嗎？

下一格幫你算第 2 題。

In [ ]:
# 驗算：這是「差一點」，還是「從頭就不可能」？
params = 10_737_395_235          # M1 那格印過的數字
T4     = 14.56                   # OOM 訊息裡的 total capacity，單位 GiB

fp16 = params * 2   / 1024**3
q4   = params * 0.5 / 1024**3
print(f"16-bit 權重（load_in_4bit=False）= {fp16:.2f} GiB")
print(f"4-bit  權重（load_in_4bit=True） = {q4:.2f} GiB")
print(f"T4 總容量 = {T4:.2f} GiB")
print()
print(f"OOM 當下已配置 14.40 GiB，最後要不到的是 28 MiB（= {28/1024:.3f} GiB）。")
print(f"但真正的缺口是 {fp16 - T4:.2f} GiB —— 模型是這張卡的 {fp16 / T4:.2f} 倍。")
print()
print("注意單位：M1 那張表寫 fp16 = 21.5 GB，用的是十進位 GB；")
print(f"這裡跟 OOM 訊息一致用 GiB，同一個數字寫成 {fp16:.2f}。兩個都對，別看到不一樣就慌。")

### 答案

1. **前提不成立。** 那行的條件是「if reserved but unallocated memory **is large**」——
   往上兩行，它是 **21.48 MiB**。一點都不大。沒有東西可以回收，
   `expandable_segments` 在這裡**完全無效**。它治的是碎片化，我們的病不是碎片化。
2. **不是。** 缺口是 5 GiB 以上，不是 28 MiB。
   OOM 訊息永遠只報「最後那一次要不到的配置」，那個數字跟你差多少**完全無關**——
   它是壓垮駱駝的最後一根稻草，不是駱駝少的那幾公斤。
   看 `14.40 GiB is allocated` 貼著 `total capacity 14.56 GiB`，才知道是「整張卡被填滿」。
3. **不會。** `device_map="auto"` 要真的能 offload，得同時給 `max_memory` 和 `offload_folder`；
   兩個都沒給的時候它沒地方放，unsloth 也會把裝置圖蓋成 `{"": 0}`（全部塞 GPU 0）。
   **那行註解描述的事情從來沒發生過。**

三題的共同點：**訊息和註解都在對你說話，但它們都不知道你的情況。**

> 這就是**守則二**的延伸——「相信執行輸出，不要相信 README；連執行輸出也要驗算」。
> 錯誤訊息附的建議，跟 README 一樣是別人先寫好的通用模板，**也是一個待驗證的假設**。

### 那正確的做法是什麼

只有兩條路，選一條：

| 做法 | 怎麼寫 | 適用 |
|---|---|---|
| **不要合併**（預設） | 載 4-bit base，再掛 adapter | 部署、Gradio、HF Space |
| 合併了、但要塞進小卡 | 把合併結果用 `load_in_4bit=True` 載回來 | 已經有合併檔又只有 T4 |

第二條會**量化兩次**（訓練時 4-bit → 合併成 16-bit → 再壓回 4-bit），拿來 demo 可以，
拿來報分數不行。所以 M6-1 那張表才說：**預設就只存 adapter。**

> 註：這門課發下去的 repo，那一格**已經改成 `load_in_4bit=True`** 了。
> 上面引的是**原始版本**——想自己重現這個錯誤，把它改回 `False` 就看得到。

---
## M6-3｜用 Gradio 試試你的模型

注意這個介面**直接呼叫 M5-2 定義的 `generate_caption`**——沒有第二份推論邏輯。

這就是守則四的具體長相：**評估用哪段程式碼，部署就用哪段。**
專案原本的 `app.py` 兩邊各寫一份，結果部署處理了 prompt 回音、評估忘了處理。

In [ ]:
!pip install -q gradio
import gradio as gr

def analyze(img, prompt, temperature, max_tokens):
    if img is None:
        return "請先上傳一張圖片"
    return generate_caption(                 # ← 與評估完全同一段程式碼
        img.convert("RGB"),
        prompt=prompt.strip() or INSTRUCTION,
        max_new_tokens=int(max_tokens),
        do_sample=temperature > 0,           # 明寫，不依賴預設值（M5-4 的教訓）
        temperature=max(temperature, 1e-5),
    )

gr.Interface(
    fn=analyze,
    inputs=[gr.Image(type="pil", label="天文影像"),
            gr.Textbox(label="Prompt（留空用預設）", placeholder=INSTRUCTION),
            gr.Slider(0.0, 1.5, value=0.0, step=0.1, label="Temperature（0 = greedy）"),
            gr.Slider(32, 256, value=128, step=32, label="最多產生幾個 token")],
    outputs=gr.Textbox(label="模型輸出", lines=6),
    title="AstroVision（你剛訓練的模型）",
    flagging_mode="never",
).launch(share=True, debug=False)

點畫面裡的公開網址就能用手機開。**把 temperature 從 0 拉到 1.5 試試看**——
你會直接看到 M5-4 講的：溫度越高越有創意，也越容易亂講。captioning 該用低溫。

> 用完記得回到上面那格按停止，Colab 的 GPU 額度是有限的。

---
## M7｜五條守則與作業

```
1. 微調是在教「說話方式」，不是在教「知識」。
2. 相信執行輸出，不要相信 README；連執行輸出也要驗算。
3. 分析用的清洗 ≠ 訓練用的清洗。
4. 評估路徑跟部署路徑，必須是同一段程式碼。
5. 一個有 bug 的評估數字，比沒有數字更危險。
```

我們今天在一份**能跑、公開發表、看起來很專業**的 code 裡找到四個真實的 bug。
而找到它們靠的不是讀 code，是**看到一個怪怪的數字，然後不放過它**——`length_ratio = 2.05`，那就是全部的線索。

> 你們以後最有價值的能力，不是會呼叫 `FastVisionModel.from_pretrained`（那查文件就會了），
> 而是：**當一個數字看起來怪怪的時候，你會停下來。**

---

### 作業

**Part A 必做**　A1 修資料洩漏並重訓 ・ A2 修 prompt 回音並回報前後對照 ・
A3 把推論抽成一個函式讓評估與部署共用 ・ A4 加上 validation loss 並判斷過擬合

**Part B 選一**　B1 視覺塔該不該訓練（消融）・ B2 清洗策略對照 ・
B3 解碼溫度掃描 ・ B4 換指標（CLIPScore / LLM-as-judge）

**Part C 加分**　驗證 `app.py` 的 `get_peft_model(model, lora_adapter=...)` 是否真的載入了訓練好的權重。
**要有證據**（印 LoRA 權重統計量，或做 base model 對照）。

評分重點**不是分數變高**，是你有沒有**誠實說明每個數字的來源與限制**。